# Overview

This notebook is to go through the issues that have the previous Sharepoint and update them to the new location.

Just for context, when we started this project we store most of our files in Prosperity Sharepoint because they started the project of Prosperity360 (or something like that). Now, almost two years later we all lost momentarily access to that location, so Lorena decided to move everything to our Sharepoint so we can rely on constant access from our team members. As a consequence we have to update these link across all scripts and documentation we have.hasattr

This is one of those efforts.

# Setup

In [ ]:
import requests
import os
import time
import json

## Load token stored locally with `load_token`

In [ ]:
def load_token(file_path, key="WB_DECIS_TOKEN"):
    try:
        with open(file_path, "r") as f:
            for line in f:
                if "=" in line:
                    temp_key, value = line.strip().split("=", 1)
                    if temp_key.strip() == key:
                        return value.strip()
        raise ValueError("The token key you provided was not found")
    except FileNotFoundError as e:
        raise FileNotFoundError(f"File not found... {e}")

In [ ]:
token = load_token("../../github.token")

In [ ]:
# Set up headers with the token
headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/vnd.github+json",
}

In [ ]:
# token = load_token("../../github.token")
# owner = "WB-DECIS"
# repo = "testing_issues"
token = load_token("../../github.token", key="DATA360_DATABRICKS")
owner = "worldbank"
repo = "data360-pipelines-databricks"

## Get list of issues with Sharepoint links to update

In [ ]:
# GitHub API endpoint (example: get authenticated user info)
url = f"https://api.github.com/repos/{owner}/{repo}/issues"

In [ ]:
# params
params = {
	"state": "all",  # Options: open, closed, all
	"per_page": 100,  # Number of results per page (max 100)
	"page": 1,       # Page number to retrieve
}

In [ ]:
# Iterate through pages
all_issues = []
while True:
	response = requests.get(url, headers=headers, params=params)
	if response.status_code != 200:
		print(f"Failed to retrieve issues: {response.status_code}")
		break

	issues = response.json()
	if not issues:
		break

	all_issues.extend(issues)
	params["page"] += 1
	time.sleep(1)  # To avoid hitting rate limits

In [ ]:
print(len(all_issues))
all_issues

In [ ]:
# Filter only issues with "Data Modeling - Curator" in the title
modeling_issues = [issue for issue in all_issues if "- Data modeling - Curator" in issue.get('title')]
print(len(modeling_issues))
modeling_issues

In [ ]:
# New body text
body = "- [ ] Evaluation of the data to identify dimensionality, attributes and collapsing options.\n- [ ] Re-modeling design to comply with the established standards (DSDs, codelists, etc.)\n- [ ] Document finalized dimensions and items code(s) and name(s) in the template.\n\nTo perform this task, create a copy of this template [00. MAPPING_TEMPLATE.xlsx](https://worldbankgroup.sharepoint.com/:x:/r/sites/ddh2/Shared%20Documents/Data360/Data%20Management/Mappings/00.%20MAPPING_TEMPLATE.xlsx?d=wc4e5f31fb5634436b593f3d7c2a07d10&csf=1&web=1&e=wgak8E), complete the modeling documentation and deposit the new file in this [location](https://worldbankgroup.sharepoint.com/:f:/r/sites/ddh2/Shared%20Documents/Data360/Data%20Management/Mappings?csf=1&web=1&e=E1OETQ) \n\n**Note: When adding comments, please add as a title the item that you are referring to. Example:**\n```\n# [Item title]\n\n[Your comment, e.g., This was done locally and ...]\n```\n"

In [ ]:
# Loop through each issue and update links
for issue in modeling_issues:
	print(issue['number'], " - ", issue['title'])
	issue_number = issue['number']
	
	# GitHub API endpoint to post a comment on the issue
	comment_url = f"https://api.github.com/repos/{owner}/{repo}/issues/{issue_number}"
		
	# Post the comment
	response = requests.patch(
		comment_url, 
		headers=headers, 
		json={"body": body}
	)
		
	print(">>>>>>> ", response.status_code)

	if response.status_code in [200, 201]:
		print(f"Comment added to issue #{issue_number}")
	else:
		print(f"Failed to add comment to issue #{issue_number}: {response.status_code}")

In [ ]:
response.status_code

In [ ]:
response.text